In [1]:
#use environment.yml
import pandas as pd
import numpy as np
import http.client
import requests
import json
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os
import holidays
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor
)
import joblib
import mlflow
import mlflow.sklearn
from pathlib import Path
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from scipy.sparse import csr_matrix

from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient


Prendre top 20stations

In [37]:
#Charger données
PATH = "../01_Data/"
dataset = pd.read_parquet(f"{PATH}dataset_top20_stations.parquet")

dataset.describe().T
dataset.info()
dataset.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175200 entries, 0 to 175199
Data columns (total 23 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   station_id                 175200 non-null  object        
 1   date                       175200 non-null  datetime64[ns]
 2   date_hour                  175200 non-null  datetime64[ns]
 3   year                       175200 non-null  int32         
 4   month                      175200 non-null  int32         
 5   day                        175200 non-null  int32         
 6   jour_semaine               175200 non-null  object        
 7   hour                       175200 non-null  int64         
 8   num_bikes_taken            175200 non-null  int64         
 9   num_bikes_dropped          175200 non-null  int64         
 10  net_flow                   175200 non-null  int64         
 11  temp                       175200 non-null  float32 

,station_id,date,date_hour,year,month,day,jour_semaine,hour,num_bikes_taken,num_bikes_dropped,...,precipitation_total,average_wind_speed,coco,is_holiday,coco_label,coco_group,precipitation_total_round,precipitation_total_bin,temp_round,temp_bin
0,5374.01,2024-11-01,2024-11-01 00:00:00,2024,11,1,Vendredi,0,11,10,...,0.0,7.000000,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",23.0,"(22.3, 24.75]"
1,5374.01,2024-11-01,2024-11-01 01:00:00,2024,11,1,Vendredi,1,16,11,...,0.0,15.000000,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",24.0,"(22.3, 24.75]"
2,5374.01,2024-11-01,2024-11-01 02:00:00,2024,11,1,Vendredi,2,10,5,...,0.0,7.000000,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",23.0,"(22.3, 24.75]"
3,5374.01,2024-11-01,2024-11-01 03:00:00,2024,11,1,Vendredi,3,4,3,...,0.0,22.700001,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",23.0,"(22.3, 24.75]"
4,5374.01,2024-11-01,2024-11-01 04:00:00,2024,11,1,Vendredi,4,1,3,...,0.0,11.000000,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",22.0,"(19.85, 22.3]"


### Features engineering
Prédiction Random forest avec données temporelles -> apprentissage supervisé
Random Forest supoose que les observations sont indépendantes
-> création de lags: Utiliser les valeurs passées pour prédire la valeur future.
-> Fonctionnalités temporelles : Extraire des informations de la date (Jour de la semaine, mois, année, jour de l'année, trimestre, vacances).
-> Statistiques glissantes (Rolling statistics) : Calculer des moyennes ou des écarts-types sur une fenêtre de temps glissante (ex: moyenne des 7 derniers jours)



In [ ]:
def prepare_features(dataset, month_studied= None):
    dataset = dataset.copy()
    
    dataset["date"] = pd.to_datetime(dataset["date"], format="%Y-%m-%d", errors="raise")
    dataset = dataset.sort_values(["station_id","date_hour"])
            
    # --- Filtrer les dates si nécessaire ---
    if month_studied is not None:           
        start = pd.to_datetime(month_studied[0] + "-01")
        end   = pd.to_datetime(month_studied[1] + "-01") + pd.offsets.MonthEnd(1)

        dataset = dataset[
            (dataset["date"] >= start) &
            (dataset["date"] <= end)
        ]
    # Target (t+1h) + time-series features
    dataset["y"] = dataset.groupby("station_id")["net_flow"].shift(-1)
    
    #Features temporelles supplémentaires
    dataset['hour_sin'] = np.sin(2*np.pi*dataset['hour']/24)
    dataset['hour_cos'] = np.cos(2*np.pi*dataset['hour']/24)
    dataset['day_sin'] = np.sin(2*np.pi*dataset['day']/31)
    dataset['day_cos'] = np.cos(2*np.pi*dataset['day']/31)
    dataset['month_sin'] = np.sin(2*np.pi*dataset['month']/12)
    dataset['month_cos'] = np.cos(2*np.pi*dataset['month']/12)
    dataset['is_weekend'] = dataset['jour_semaine'].isin(['Samedi','Dimanche']).astype(int)
    dataset['is_peak'] = (((dataset['hour']>=6) & (dataset['hour']<10)) | ((dataset['hour']>=16) & (dataset['hour']<20))).astype(int)
    
    
    # --- Features météo ---
    dataset['cold_weather'] = (dataset['temp']<5).astype(int)
    dataset['hot_weather'] = (dataset['temp']>30).astype(int)
    dataset['heavy_rain'] = (dataset['precipitation_total']>5).astype(int)
    
    # --- Lag features ---
    
    lags = [1,2,24]
    for l in lags:
        dataset[f"net_flow_lag_{l}"] = dataset.groupby("station_id")["net_flow"].shift(l)

    dataset["net_flow_roll_3"] = dataset.groupby("station_id")["net_flow"].transform(
        lambda x: x.shift(1).rolling(3).mean()
    )

    dataset["net_flow_roll_24"] = dataset.groupby("station_id")["net_flow"].transform(
        lambda x: x.shift(1).rolling(24).mean()
    )
        
    # optional lags for pickups/drops (if present)
    for col in ["num_bikes_taken", "num_bikes_dropped"]:
        if col in dataset.columns:
            dataset[f"{col}_lag_1"] = dataset.groupby("station_id")[col].shift(1)

    # --- Features finales pour le modèle ---
    features = [
        #'hour_sin','hour_cos','day_sin','day_cos','month_sin','month_cos',
        #'is_weekend','is_peak',
        "station_id", 'year', 'month', 'day', 'hour',
        'temp','precipitation_total','relative_humidity','average_wind_speed',
        #'cold_weather','hot_weather','heavy_rain',
        'num_bikes_taken_lag_1','num_bikes_dropped_lag_1',
        'net_flow_lag_1','net_flow_lag_2','net_flow_lag_24','net_flow_roll_3','net_flow_roll_24',
        'jour_semaine', 'coco_group', 'is_holiday', 'coco'
        
    ]
    
    # Drop lignes avec NaN issues des lags
    needed = ["y"] + [f"net_flow_lag_{l}" for l in lags] + ["net_flow_roll_3","net_flow_roll_24"]
    dataset = dataset.dropna(subset=needed).reset_index(drop=True)
    
    # Cible
    target = 'net_flow'
    
    cols_to_keep = [col for col in features if col in dataset.columns]
    if 'net_flow' not in cols_to_keep:
        cols_to_keep.append('net_flow')
        
    filtered_df = dataset[cols_to_keep].copy()
    
    return filtered_df, features, target


dataset_all, features, target = prepare_features(dataset)#, month_studied=month_studied)


In [39]:
# Fonction de normalisation
def normalize_station(s):
    s = str(s).strip()
    if '.' in s:
        integer, decimal = s.split('.')
        decimal = decimal.rstrip('0') or '0'   # supprime les zéros finaux
        return f"{integer}.{decimal}"
    return s

TOP20_STATION_LIST = PATH +'top20_station_list.csv'
top20_df = pd.read_csv(TOP20_STATION_LIST, header=None, names=["station_id"])
top20_df["station_id"] = top20_df["station_id"].astype(str).str.strip()
print(f"Nb stations : {len(top20_df)}") 

top20_df["station_id"] = top20_df["station_id"].apply(normalize_station)
top20_station = top20_df["station_id"].tolist()

# Normalise le dataset
dataset_all["station_id"] = dataset_all["station_id"].apply(normalize_station)

# Vérifie
manquantes = set(top20_station) - set(dataset_all["station_id"].unique())
print("Stations manquantes après fix :", manquantes)  # doit être vide

Nb stations : 20
Stations manquantes après fix : set()


In [40]:
FILE = "data/dataset_station_preprocessed.parquet"
if FILE in os.listdir():
        os.remove(FILE)
dataset_all.to_parquet(FILE, index=False, compression="snappy")

PATH="../data"
if FILE in os.listdir():
        os.remove(FILE)
dataset_all.to_parquet(FILE, index=False, compression="snappy")

dataset_fe = dataset_all[dataset_all["station_id"].isin(top20_station)].copy()
print(f"Stations dans dataset_fe : {dataset_fe['station_id'].nunique()}")
print(f"Lignes : {len(dataset_fe)}")

TOP20_FILE = "data/dataset_top20_stations.parquet"
if TOP20_FILE in os.listdir():
        os.remove(TOP20_FILE)
dataset_fe.to_parquet(TOP20_FILE, index=False, compression="snappy")

print("Nombre de lignes :", len(dataset_fe))

Stations dans dataset_fe : 20
Lignes : 174700
Nombre de lignes : 174700


Split train/test pour données temporelles:
- pas de validation croisée aléatoire (K-Fold classique). Il faut respecter la structure temporelle. Le jeu d'entrainement doit précéder le jeu de test

In [42]:
# Split temporel correct — garde toutes les stations dans train et test
# Coupe sur la date plutôt que sur l'index global
dataset_fe = dataset_fe.sort_values(["year", "month", "day", "hour"])
cut = int(len(dataset_fe) * 0.8)
train_df = dataset_fe.iloc[:cut].copy()
test_df  = dataset_fe.iloc[cut:].copy()

print(f"Stations train : {train_df['station_id'].nunique()}")
print(f"Stations test  : {test_df['station_id'].nunique()}")

X_train = train_df[features]
y_train = train_df[target]
X_test  = test_df[features]
y_test  = test_df[target]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# Baselin
# FIX: shift(1) par station pour éviter la fuite entre stations
# net_flow(t) ≈ net_flow(t-1) — premier enregistrement de chaque station → 0
y_pred_baseline = (
    test_df
    .groupby("station_id", observed=True)["net_flow"]
    .shift(1)
    .fillna(0)
    .to_numpy()
)
rmse_baseline = float(np.sqrt(mean_squared_error(y_test, y_pred_baseline)))
mae_baseline  = float(mean_absolute_error(y_test, y_pred_baseline))
r2_baseline   = float(r2_score(y_test, y_pred_baseline))

#Preprocessing
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype) == "category"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, num_cols),
        ("cat", categorical_preprocess, cat_cols),
    ],
    remainder="drop"
)

Stations train : 20
Stations test  : 20
Train shape: (139760, 21), Test shape: (34940, 21)


In [45]:
#Models
models = [
    ("LinearRegression", LinearRegression()),
    ("Ridge(alpha=1.0)", Ridge(alpha=1.0, random_state=42)),
    ("Lasso(alpha=0.001)", Lasso(alpha=0.001, random_state=42, max_iter=20000)),
    ("RandomForest(300,depth=14)", RandomForestRegressor(
        n_estimators=300, max_depth=14, random_state=42, n_jobs=-1
    )),
    ("ExtraTrees(500,depth=16)", ExtraTreesRegressor(
        n_estimators=500, max_depth=16, random_state=42, n_jobs=-1
    )),
    ("XGBoost", XGBRegressor(
        n_estimators=600, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.0, reg_lambda=1.0,
        random_state=42, n_jobs=-1
    )),
    ("LightGBM", LGBMRegressor(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbose=-1,        # silence les logs d'entraînement
    )),
]

def score(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    return rmse, mae, r2

In [46]:
#Train + score
results = []
X_train_prep = csr_matrix(preprocess.fit_transform(X_train))
X_test_prep  = csr_matrix(preprocess.transform(X_test))

for name, model in models:
    print(f"  Training {name}...")
    model.fit(X_train_prep, y_train)
    pred = model.predict(X_test_prep)
    rmse, mae, r2 = score(y_test, pred)
    results.append({
        "model":              name,
        "rmse":               rmse,
        "mae":                mae,
        "r2":                 r2,
        "rmse_gain_vs_baseline": rmse_baseline - rmse,
    })

#Tableau comparatif benchmark
rows_bench = [{
    "Modèle":        "Baseline (net_flow t-1)",
    "RMSE":          rmse_baseline,
    "MAE":           mae_baseline,
    "R²":            r2_baseline,
    "Gain vs Baseline": 0.0,
    "Bat le Baseline":  "—",
}]
for r in results:
    rows_bench.append({
        "Modèle":        r["model"],
        "RMSE":          r["rmse"],
        "MAE":           r["mae"],
        "R²":            r["r2"],
        "Gain vs Baseline": rmse_baseline - r["rmse"],
        "Bat le Baseline":  "✅" if r["rmse"] < rmse_baseline else "❌",
    })

df_bench = (
    pd.DataFrame(rows_bench)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
df_bench["RMSE"]          = df_bench["RMSE"].map("{:.4f}".format)
df_bench["MAE"]           = df_bench["MAE"].map("{:.4f}".format)
df_bench["R²"]            = df_bench["R²"].map("{:.4f}".format)
df_bench["Gain vs Baseline"] = df_bench["Gain vs Baseline"].map("{:+.4f}".format)

print("\n=== Benchmark — Tous les modèles (trié par RMSE) ===")
print(df_bench.to_string(index=False))

  Training LinearRegression...
  Training Ridge(alpha=1.0)...
  Training Lasso(alpha=0.001)...
  Training RandomForest(300,depth=14)...
  Training ExtraTrees(500,depth=16)...
  Training XGBoost...
  Training LightGBM...

=== Benchmark — Tous les modèles (trié par RMSE) ===
                    Modèle    RMSE    MAE      R² Gain vs Baseline Bat le Baseline
                   XGBoost  6.4864 4.3528  0.5305          +4.3574               ✅
  ExtraTrees(500,depth=16)  6.5262 4.3389  0.5248          +4.3176               ✅
                  LightGBM  6.5635 4.3836  0.5193          +4.2803               ✅
RandomForest(300,depth=14)  6.7850 4.4778  0.4863          +4.0588               ✅
        Lasso(alpha=0.001)  7.7441 5.0819  0.3308          +3.0998               ✅
          LinearRegression  7.7442 5.0842  0.3308          +3.0996               ✅
          Ridge(alpha=1.0)  7.7447 5.0857  0.3307          +3.0991               ✅
   Baseline (net_flow t-1) 10.8439 6.9569 -0.3121          +0.

In [ ]:
# Hyperparameter tuning XGBoost
xgb_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", XGBRegressor(
        random_state=42,
        n_jobs=1, # n_jobs=1 car RandomizedSearchCV parallélise les fits
    ))
])

param_dist_xgb_pipe = {
    "model__n_estimators":     [400, 600, 800, 1000, 1200],
    "model__max_depth":        [3, 4, 5, 6, 8],
    "model__learning_rate":    [0.01, 0.03, 0.05, 0.1],
    "model__subsample":        [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__reg_alpha":        [0.0, 0.1, 1.0],
    "model__reg_lambda":       [0.5, 1.0, 2.0, 5.0],
    "model__gamma":            [0.0, 0.1, 0.5, 1.0],
}


tscv = TimeSeriesSplit(n_splits=5)
search = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=param_dist_xgb_pipe,
    n_iter=50,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    verbose=1,
    n_jobs=-1,             # parallélise les fits CV
    random_state=42,
)

search.fit(X_train, y_train)

best_xgb  = search.best_estimator_
pred_best = best_xgb.predict(X_test)
rmse_best, mae_best, r2_best = score(y_test, pred_best)

#Tableau final
rows_final = [{
    "Modèle":           "Baseline (net_flow t-1)",
    "RMSE":             rmse_baseline,
    "MAE":              mae_baseline,
    "R²":               r2_baseline,
    "Gain vs Baseline": 0.0,
    "Bat le Baseline":  "—",
}]
for r in results:
    rows_final.append({
        "Modèle":           r["model"],
        "RMSE":             r["rmse"],
        "MAE":              r["mae"],
        "R²":               r["r2"],
        "Gain vs Baseline": rmse_baseline - r["rmse"],
        "Bat le Baseline":  "✅" if r["rmse"] < rmse_baseline else "❌",
    })
rows_final.append({
    "Modèle":           "XGBRegressor tuné (RandomizedSearchCV)",
    "RMSE":             rmse_best,
    "MAE":              mae_best,
    "R²":               r2_best,
    "Gain vs Baseline": rmse_baseline - rmse_best,
    "Bat le Baseline":  "✅" if rmse_best < rmse_baseline else "❌",
})

df_final = (
    pd.DataFrame(rows_final)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
df_final["RMSE"]             = df_final["RMSE"].map("{:.4f}".format)
df_final["MAE"]              = df_final["MAE"].map("{:.4f}".format)
df_final["R²"]               = df_final["R²"].map("{:.4f}".format)
df_final["Gain vs Baseline"] = df_final["Gain vs Baseline"].map("{:+.4f}".format)

print(f"\nBest params: {search.best_params_}")
print(f"Best CV RMSE: {-search.best_score_:.4f}")
print("\n=== Tableau final — Tous les modèles + XGBRegressor tuné (trié par RMSE) ===")
print(df_final.to_string(index=False))

Fitting 5 folds for each of 50 candidates, totalling 250 fits



Best params: {'model__subsample': 0.8, 'model__reg_lambda': 2.0, 'model__reg_alpha': 0.0, 'model__n_estimators': 1000, 'model__min_child_weight': 5, 'model__max_depth': 6, 'model__learning_rate': 0.05, 'model__gamma': 0.0, 'model__colsample_bytree': 0.8}
Best CV RMSE: 5.4303

=== Tableau final — Tous les modèles + XGBRegressor tuné (trié par RMSE) ===
                              Modèle    RMSE    MAE      R² Gain vs Baseline Bat le Baseline
ExtraTrees tuné (RandomizedSearchCV)  6.4257 4.3173  0.5393          +4.4181               ✅
                             XGBoost  6.4864 4.3528  0.5305          +4.3574               ✅
            ExtraTrees(500,depth=16)  6.5262 4.3389  0.5248          +4.3176               ✅
                            LightGBM  6.5635 4.3836  0.5193          +4.2803               ✅
          RandomForest(300,depth=14)  6.7850 4.4778  0.4863          +4.0588               ✅
                  Lasso(alpha=0.001)  7.7441 5.0819  0.3308          +3.0998           

### Sauvegarde modèle

In [49]:
final_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        n_estimators=search.best_params_['model__n_estimators'],
        max_depth=search.best_params_['model__max_depth'],
        learning_rate=search.best_params_['model__learning_rate'],
        subsample=search.best_params_['model__subsample'],
        colsample_bytree=search.best_params_['model__colsample_bytree'],
        reg_alpha=search.best_params_['model__reg_alpha'],
        reg_lambda=search.best_params_['model__reg_lambda'],
        min_child_weight=search.best_params_['model__min_child_weight'],
        gamma=search.best_params_['model__gamma'],
        random_state=42,
        n_jobs=-1,
    ))
])
final_pipe.fit(X_train, y_train)
 
# Vérification finale sur le test set
pred_final = final_pipe.predict(X_test)
rmse_f, mae_f, r2_f = score(y_test, pred_final)

print(f"Pipeline final — RMSE: {rmse_f:.4f} | MAE: {mae_f:.4f} | R²: {r2_f:.4f}")
 
#Sauvegarde locale
Path("model").mkdir(exist_ok=True)
model_path = "model/citibike_forecast_model.joblib"
joblib.dump(final_pipe, model_path)
print("Pipeline sauvegardé localement")

# Vérifie la taille
size_mb = os.path.getsize(model_path) / 1e6
print(f"Pipeline sauvegardé — Taille : {size_mb:.1f} MB")


Pipeline final — RMSE: 6.4257 | MAE: 4.3173 | R²: 0.5393
Pipeline sauvegardé localement
Pipeline sauvegardé — Taille : 4.0 MB


In [52]:
# Calcul erreur par station
df_errors_station = pd.DataFrame({
    "y_true":   y_test.values,
    "y_pred":   pred_final,
    "station":  X_test["station_id"].values,
})

errors_by_station = (
    df_errors_station
    .groupby("station")
    .apply(lambda g: pd.Series({
        "rmse": float(np.sqrt(mean_squared_error(g["y_true"], g["y_pred"]))),
        "mae":  float(mean_absolute_error(g["y_true"], g["y_pred"])),
        "bias": float((g["y_pred"] - g["y_true"]).mean()),
        "n_obs": len(g),
    }), include_groups=False)
    .reset_index()
    .sort_values("rmse", ascending=False)
)

print(errors_by_station.to_string(index=False))

worst_stations = errors_by_station.head(5)
best_stations  = errors_by_station.tail(5)

print("\n🔴 Stations les plus difficiles :")
print(worst_stations[["station", "rmse", "mae", "bias"]].to_string(index=False))
print("\n🟢 Stations les mieux prédites :")
print(best_stations[["station",  "rmse", "mae", "bias"]].to_string(index=False))

#Visualisation erreur par station
fig = go.Figure()

# Couleur selon le biais : rouge = sur-estime, bleu = sous-estime
colors = ["#E24B4A" if b > 0 else "#378ADD" for b in errors_by_station["bias"]]

# Barres RMSE triées
fig.add_trace(go.Bar(
    x=errors_by_station["station"],
    y=errors_by_station["rmse"],
    name="RMSE",
    marker_color=colors,
    opacity=0.85,
    text=errors_by_station["rmse"].map("{:.2f}".format),
    textposition="outside",
    textfont=dict(size=11),
))

# MAE en ligne
fig.add_trace(go.Scatter(
    x=errors_by_station["station"],
    y=errors_by_station["mae"],
    name="MAE",
    mode="lines+markers",
    line=dict(color="#EF9F27", width=2),
    marker=dict(size=7),
))

# Ligne RMSE moyen global
rmse_mean = errors_by_station["rmse"].mean()
fig.add_hline(
    y=rmse_mean,
    line_dash="dash",
    line_color="gray",
    line_width=1.5,
    annotation_text=f"RMSE moyen : {rmse_mean:.2f}",
    annotation_position="top right",
    annotation_font_size=11,
)

fig.update_layout(
    title="Erreur du modèle par station (RMSE)",
    xaxis_title="Station",
    yaxis_title="Erreur (vélos)",
    xaxis_tickangle=-45,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    template="plotly_white",
    height=500,
    bargap=0.3,
    annotations=[
        dict(
            x=0.01, y=0.97, xref="paper", yref="paper",
            text="🔴 Biais positif (sur-estime)  🔵 Biais négatif (sous-estime)",
            showarrow=False, font=dict(size=11), align="left",
        )
    ]
)

fig.show()

#Scatter biais vs RMSE
fig2 = px.scatter(
    errors_by_station,
    x="bias",
    y="rmse",
    text="station",
    color="rmse",
    color_continuous_scale="RdYlGn_r",
    size="n_obs",
    title="Biais vs RMSE par station",
    labels={"bias": "Biais moyen (pred - réel)", "rmse": "RMSE"},
    template="plotly_white",
    height=450,
)
fig2.update_traces(textposition="top center", textfont_size=10)
fig2.add_vline(x=0, line_dash="dash", line_color="gray", line_width=1)
fig2.update_coloraxes(showscale=False)
fig2.show()

station     rmse      mae      bias  n_obs
6492.08 9.382582 5.553994 -0.313174 1747.0
6450.05 7.460907 5.009783 -0.124887 1747.0
6331.01 7.208305 5.021570 -0.223011 1747.0
5788.13 7.177520 5.021603 -0.073438 1747.0
6173.08 6.765659 4.324092 -0.028379 1747.0
6197.08 6.640230 4.670362  0.078420 1747.0
6140.05 6.639857 4.591740 -0.268250 1747.0
6364.07 6.454439 4.561102  0.356786 1747.0
5905.12 6.432958 4.402042 -0.528618 1747.0
 5980.1 6.335809 4.212464 -0.335432 1747.0
5374.01 6.319637 4.318673 -0.924576 1747.0
6233.04 6.062437 4.099773  0.095395 1747.0
 6948.1 6.054366 4.129149 -0.022884 1747.0
6459.07 6.001215 4.104648  0.182606 1747.0
6726.01 5.807290 3.960899 -0.020305 1747.0
6072.06 5.761204 4.085901  0.372398 1747.0
5905.14 5.571087 3.887240 -0.080194 1747.0
 6098.1 5.251710 3.473351 -0.154769 1747.0
5779.11 4.904557 3.544331 -0.100686 1747.0
5492.05 4.746941 3.372470 -0.116123 1747.0

🔴 Stations les plus difficiles :
station     rmse      mae      bias
6492.08 9.382582 5.553994 -

In [51]:
# Calcul erreur par heure
pred_final = final_pipe.predict(X_test)

df_errors = pd.DataFrame({
    "y_true":  y_test.values,
    "y_pred":  pred_final,
    "hour":    X_test["hour"].values,
})

errors_by_hour = (
    df_errors
    .groupby("hour")
    .apply(lambda g: pd.Series({
        "rmse":    float(np.sqrt(mean_squared_error(g.y_true, g.y_pred))),
        "mae":     float(mean_absolute_error(g.y_true, g.y_pred)),
        "n_obs":   len(g),
        "bias":    float((g.y_pred - g.y_true).mean()),  # erreur systématique
    }))
    .reset_index()
    .sort_values("hour")
)

print(errors_by_hour.to_string(index=False))

#Visualisation
fig = go.Figure()

# RMSE par heure
fig.add_trace(go.Bar(
    x=errors_by_hour["hour"],
    y=errors_by_hour["rmse"],
    name="RMSE",
    marker_color="steelblue",
    opacity=0.8,
))

# MAE par heure (ligne)
fig.add_trace(go.Scatter(
    x=errors_by_hour["hour"],
    y=errors_by_hour["mae"],
    name="MAE",
    mode="lines+markers",
    line=dict(color="coral", width=2),
    marker=dict(size=6),
))

# Biais (ligne pointillée)
fig.add_trace(go.Scatter(
    x=errors_by_hour["hour"],
    y=errors_by_hour["bias"],
    name="Biais (pred - réel)",
    mode="lines",
    line=dict(color="gray", width=1.5, dash="dot"),
))

# Ligne zéro pour le biais
fig.add_hline(y=0, line_dash="dash", line_color="lightgray", line_width=1)

fig.update_layout(
    title="Erreur du modèle par heure de la journée",
    xaxis_title="Heure",
    yaxis_title="Erreur (vélos)",
    xaxis=dict(tickmode="linear", tick0=0, dtick=1),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    template="plotly_white",
    height=450,
)

fig.show()

#Résumé
worst  = errors_by_hour.nlargest(3, "rmse")[["hour", "rmse", "mae", "bias"]]
best   = errors_by_hour.nsmallest(3, "rmse")[["hour", "rmse", "mae", "bias"]]

print("\n🔴 Heures les plus difficiles (RMSE le plus élevé) :")
print(worst.to_string(index=False))
print("\n🟢 Heures les mieux prédites (RMSE le plus faible) :")
print(best.to_string(index=False))

 hour      rmse      mae  n_obs      bias
    0  3.643388 2.394363 1440.0  0.062367
    1  2.702403 1.821137 1440.0  0.136919
    2  2.048252 1.360928 1440.0 -0.094626
    3  1.576515 1.044124 1440.0  0.086393
    4  1.669446 1.160727 1460.0 -0.129650
    5  3.070228 2.086911 1460.0 -0.195725
    6  5.193373 3.543692 1460.0 -0.322445
    7  6.057887 4.517987 1460.0 -0.376800
    8  9.214707 6.500712 1460.0 -1.076912
    9  8.482174 6.195954 1460.0 -0.760367
   10  6.371856 4.756415 1460.0 -0.338641
   11  5.858580 4.383591 1460.0 -0.052289
   12  6.266873 4.672787 1460.0 -0.245857
   13  6.447591 4.825639 1460.0  0.293384
   14  6.898857 5.258869 1460.0  0.176721
   15  7.679780 5.720746 1460.0 -0.010559
   16  8.539932 6.508133 1460.0  0.297990
   17 11.527515 8.189985 1460.0  0.660018
   18  9.172545 6.984106 1460.0 -0.573586
   19  7.317720 5.548797 1460.0 -0.278020
   20  6.429663 4.886702 1460.0  0.113017
   21  5.560334 4.082456 1460.0 -0.111397
   22  5.332071 3.766897 1460.0  0

/var/folders/d2/j007c_wj355g0r_h_j5t1vrc0000gn/T/ipykernel_31291/3030528656.py:13: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.




🔴 Heures les plus difficiles (RMSE le plus élevé) :
 hour      rmse      mae      bias
   17 11.527515 8.189985  0.660018
    8  9.214707 6.500712 -1.076912
   18  9.172545 6.984106 -0.573586

🟢 Heures les mieux prédites (RMSE le plus faible) :
 hour     rmse      mae      bias
    3 1.576515 1.044124  0.086393
    4 1.669446 1.160727 -0.129650
    2 2.048252 1.360928 -0.094626


In [53]:
os.environ["MLFLOW_TRACKING_URI"]    = "http://127.0.0.1:5001"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://127.0.0.1:9000"
os.environ["AWS_ACCESS_KEY_ID"]      = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"]  = "minioadmin"
os.environ["AWS_DEFAULT_REGION"]     = "us-east-1"
os.environ["NO_PROXY"]               = "127.0.0.1,localhost,minio"
os.environ["no_proxy"]               = "127.0.0.1,localhost,minio"
os.environ["HTTP_PROXY"]             = ""
os.environ["HTTPS_PROXY"]            = ""
os.environ["http_proxy"]             = ""
os.environ["https_proxy"]            = ""


# ── MLflow setup ──────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "http://127.0.0.1:5001"
EXPERIMENT_NAME     = "Citibike_forecast_training"
MODEL_NAME          = "citibike_forecast_model"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

exp = client.get_experiment_by_name(EXPERIMENT_NAME)
if exp and exp.lifecycle_stage == "deleted":
    client.restore_experiment(exp.experiment_id)
    experiment_id = exp.experiment_id
elif exp is None:
    experiment_id = client.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = exp.experiment_id

try:
    client.get_registered_model(MODEL_NAME)
except mlflow.exceptions.MlflowException:
    client.create_registered_model(MODEL_NAME)

# Charge les best_params sauvegardés
with open("model/best_params.json") as f:
    best_params = json.load(f)

# ── MLflow Run ────────────────────────────────────────────────────────────────
with mlflow.start_run(
    run_name="xgboost_tuned_1000trees",
    experiment_id=experiment_id
) as run:
    rmse, mae, r2 = score(y_test, final_pipe.predict(X_test))

    mlflow.log_metric("rmse",               rmse)
    mlflow.log_metric("mae",                mae)
    mlflow.log_metric("r2",                 r2)
    mlflow.log_metric("rmse_baseline",      rmse_baseline)
    mlflow.log_metric("gain_vs_baseline",   rmse_baseline - rmse)
    mlflow.log_params(best_params)

    X_sig     = X_test.astype({col: "float64" for col in X_test.select_dtypes("int").columns})
    signature = infer_signature(X_sig, final_pipe.predict(X_test))

    model_info = mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="model",
        signature=signature,
        registered_model_name=MODEL_NAME,
    )
    print(f"✅ Run ID : {run.info.run_id}")
    print(f"   RMSE={rmse:.4f} | MAE={mae:.4f} | R²={r2:.4f}")

#Alias staging
versions = client.search_model_versions(
    filter_string=f"name='{MODEL_NAME}'",
    order_by=["version_number DESC"],
    max_results=1,
)
version = versions[0].version
client.set_registered_model_alias(name=MODEL_NAME, alias="staging", version=version)
print(f"✅ Version {version} aliasée 'staging'")
print(f"   Signature inputs  : {model_info.signature.inputs}")
print(f"   Signature outputs : {model_info.signature.outputs}")

Registered model 'citibike_forecast_model' already exists. Creating a new version of this model...
2026/05/14 10:41:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: citibike_forecast_model, version 3


✅ Run ID : 078d08aad71d435188cf9d34102431c1
   RMSE=6.4257 | MAE=4.3173 | R²=0.5393
🏃 View run xgboost_tuned_1000trees at: http://127.0.0.1:5001/#/experiments/2/runs/078d08aad71d435188cf9d34102431c1
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2
✅ Version 3 aliasée 'staging'
   Signature inputs  : ['is_peak': double (required), 'station_id': string (required), 'year': double (required), 'month': double (required), 'day': double (required), 'hour': double (required), 'temp': float (required), 'precipitation_total': float (required), 'relative_humidity': float (required), 'average_wind_speed': float (required), 'num_bikes_taken_lag_1': double (required), 'num_bikes_dropped_lag_1': double (required), 'net_flow_lag_1': double (required), 'net_flow_lag_2': double (required), 'net_flow_lag_24': double (required), 'net_flow_roll_3': double (required), 'net_flow_roll_24': double (required), 'jour_semaine': string (required), 'coco_group': string (required), 'is_holiday': boolean (

Created version '3' of model 'citibike_forecast_model'.


# Conclusion — Modélisation CitiBike Flow Prediction

La démarche a suivi 4 étapes :

1. **Baseline** — établir une référence simple : prédire le flux de l'heure précédente (`net_flow t-1`) par station, sans fuite de données entre stations
2. **Benchmark** — comparer 7 familles de modèles
3. **Sélection** — identifier le meilleur modèle (XGBoost sur 20 stations)
4. **Tuning** — optimiser ses hyperparamètres avec `RandomizedSearchCV` + `TimeSeriesSplit`

## Résultats finaux

| Modèle | RMSE | MAE | R² | Gain vs Baseline |
|---|---|---|---|---|
| **XGBoost tuné** ✅ | **6.43** | **4.32** | **0.54** | **+4.42** |
| XGBoost benchmark | 6.49 | 4.35 | 0.53 | +4.36 |
| ExtraTrees benchmark | 6.53 | 4.34 | 0.52 | +4.32 |
| LightGBM | 6.56 | 4.38 | 0.52 | +4.28 |
| RandomForest | 6.79 | 4.48 | 0.49 | +4.06 |
| Ridge / Lasso / LinReg | ~7.74 | ~5.08 | ~0.33 | +3.10 |
| **Baseline (net_flow t-1)** | **10.84** | **6.96** | **-0.31** | **0** |

Le modèle final réduit l'erreur de **+4.42 vélos par heure** par rapport au baseline, soit une réduction de **41% du RMSE**.

## Modèle retenu

**XGBoost Regressor** dans un `sklearn Pipeline` (imputation médiane + OneHotEncoder) avec les hyperparamètres suivants :

| Hyperparamètre | Valeur | Justification |
|---|---|---|
| `n_estimators` | 1000 | Plus d'arbres = meilleure stabilité sur 20 stations |
| `max_depth` | 6 | Profondeur modérée — évite l'overfitting sur les patterns inter-stations |
| `learning_rate` | 0.05 | Taux d'apprentissage standard — bon compromis vitesse/précision |
| `subsample` | 0.8 | 80% des observations par arbre — régularisation naturelle |
| `colsample_bytree` | 0.8 | 80% des features par arbre — réduit la corrélation entre arbres |
| `reg_lambda` | 2.0 | Régularisation L2 — pénalise les poids trop importants |
| `min_child_weight` | 5 | Évite les splits sur des groupes trop petits (stations peu actives) |
| `gamma` | 0.0 | Pas de seuil de gain minimum — tous les splits informatifs conservés |

## Analyse des erreurs

L'analyse par heure révèle que le modèle est **plus difficile aux heures de pointe** :

| Heure | RMSE | Biais | Interprétation |
|---|---|---|---|
| 17h | 10.90 | +1.99 | Pointe du soir — sur-estime le flux entrant |
| 9h | 9.48 | -1.63 | Pointe du matin — sous-estime le vidage des stations |
| 3h–4h | < 2.0 | ~0 | Nuit calme — quasi pas de flux à prédire |

## Limites du modèle

- **R² = 0.54** — le modèle explique 54% de la variance. Les 46% restants correspondent à des événements imprévisibles (incidents, événements sportifs, pannes de stations) non capturés dans les features
- **Modèle global** — un seul modèle pour toutes les stations. Avec ~8 735 observations par station (~6 988 en train), un modèle par station serait sous-alimenté. Il faudrait 2–3 ans de données supplémentaires pour que cette approche soit viable
- **Lag features uniquement** — le modèle ne dispose pas de données en temps réel sur l'état du réseau (vélos disponibles en ce moment dans les stations voisines)
- **Données météo horaires** — la météo est agrégée par heure ; une résolution plus fine (15 min) améliorerait probablement les prévisions sur les heures de pluie

## Perspectives

- **Features cycliques** — encoder `hour` et `month` en sinus/cosinus pour mieux représenter la continuité temporelle aux heures de pointe
- **Feature `is_peak_hour`** — indicateur explicite des heures 7h–9h et 17h–19h, là où le modèle a le plus de mal
- **Plus de données** — avec 3 ans d'historique, un modèle par station deviendrait viable et capturerait des patterns plus fins
- **Données réseau en temps réel** — utiliser l'état actuel des stations voisines comme feature améliorerait significativement les prévisions

